```bash
conda activate
cd ~/link/other_model/learn_TACTiCS
file_name=TACTiCS_04_run
{
    jupyter-nbconvert ${file_name}.ipynb --to python
    nohup ~/apps/miniconda3/envs/TACTiCS/bin/python ${file_name}.py > nohup_${file_name} &
    sleep 10 && rm ${file_name}.py
    echo 'finish'
}
```

In [1]:
import sys
from pathlib import Path
def sys_path_show():
    print(*sys.path,sep='\n')
def sys_path_append(p):
    p = Path(p)
    assert p.exists()
    p = str(p)
    None if p in sys.path else sys.path.append(p)

sys_path_append(Path("~/link").expanduser())
sys_path_append(Path("~/link/temp/241120_other_model/learn_TACTiCS").expanduser())

In [2]:
import utils as ut
from utils.general import *
from IPython.display import display

/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/publi

In [3]:
import TACTiCS
utils = TACTiCS.utils
TACTiCSNet = TACTiCS.tactics.TACTiCSNet
TACTiCS = TACTiCS.tactics.TACTiCS
# from genes import embed_proteins, calc_dist
import torch
torch.cuda.is_available()

False

# 对于TACTiCS本地化的修改

> 追加文件 `TACTiCS/__init__.py`

TACTiCS的utils 和 我的utils重名了....可恶

```python
import TACTiCS.utils as utils
import TACTiCS.tactics
```
> 修改文件 `TACTiCS/tactics.py`

```python
# import utils # TACTiCS文件夹被视为包了, 这样时外部的utils 即我的utils
import TACTiCS.utils as utils
```


> 其他 见下文

|file|function|note||
|:-|:-|:-|:-|
|tactics|TACTiCS.\_\_init\_\_|修改读入counts的方式||
|tactics|TACTiCS.train|gpu不可用, 不使用cuda||
|tactics|TACTiCS.norm_adata|版本适用写法|输出太多版本更新提示,浏览器无法承受|
|utils|utilsget_batch|版本适用写法||

In [4]:
import os
import pickle
import torch
import torch.nn as nn
import tqdm
import numpy as np
import torch
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy

with Block("customize_TACTiCS function"):

    def customize_TACTiCS_train(self, n_epochs=200, batch_size=5000):
        # 不使用 .cuda()
        
        self.model.train()
        # self.model.cuda()
    
        weight_A = torch.Tensor([1 for _ in range(self.n_classes_A)])#.cuda()
        weight_B = torch.Tensor([1 for _ in range(self.n_classes_B)])#.cuda()
        loss_fn_A = nn.CrossEntropyLoss(label_smoothing=0.1, weight=weight_A)
        loss_fn_B = nn.CrossEntropyLoss(label_smoothing=0.1, weight=weight_B)
    
        # return utils.get_batch(self.adata_A, self.dict_A["column"], 0)
        for epoch in tqdm.tqdm(range(n_epochs)):
            for batch in range(30):
                self.optimizer.zero_grad()
                
                x_A, y_A = utils.get_batch(self.adata_A, self.dict_A["column"], batch_size)
                x_B, y_B = utils.get_batch(self.adata_B, self.dict_B["column"], batch_size)
    
                # x_A, y_A = x_A.cuda(), y_A.cuda()
                # x_B, y_B = x_B.cuda(), y_B.cuda()
    
                preds_A, preds_B, closest_dist = self.model(x_A, x_B)
    
                preds_top_A = preds_A.argmax(dim=1)
                preds_top_B = preds_B.argmax(dim=1)
    
                loss = loss_fn_A(preds_A, y_A) + loss_fn_B(preds_B, y_B) + closest_dist
                loss.backward()
                nn.utils.clip_grad_value_(self.model.parameters(), clip_value=0.5)
                self.optimizer.step()
    
            utils.update_weights_focal(y_A, preds_top_A, self.n_classes_A, weight_A)
            utils.update_weights_focal(y_B, preds_top_B, self.n_classes_B, weight_B)
    
    TACTiCS.train = customize_TACTiCS_train
    del customize_TACTiCS_train
    
    
    def customize_TACTiCS_load(self, folder):
        assert os.path.exists(folder)

        self.adata_A = ad.read_h5ad(os.path.join(folder, "species_A_pr.h5ad"))
        self.adata_B = ad.read_h5ad(os.path.join(folder, "species_B_pr.h5ad"))

        with open(os.path.join(folder, "matches.pkl"), "rb") as f:
            self.dict_A, self.dict_B, self.gene_matches = pickle.load(f)

        self.n_classes_A = len(self.adata_A.obs[self.dict_A["column"]].cat.categories)
        self.n_classes_B = len(self.adata_B.obs[self.dict_B["column"]].cat.categories)

        self.model = TACTiCSNet(self.n_classes_A, self.n_classes_B, len(self.adata_A.var_names), len(self.adata_B.var_names), self.gene_matches)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001, weight_decay=0.01)
        # 版本适用写法
        self.model.load_state_dict(torch.load(os.path.join(folder, "model.pth"),weights_only=True))
        self.optimizer.load_state_dict(torch.load(os.path.join(folder, "optim.pth"),weights_only=True))
    TACTiCS.load = customize_TACTiCS_load
    del customize_TACTiCS_load
    
    def customize_TACTiCS_norm_adata(self, adata: ad.AnnData, proteins):
        # Normalize per cell and with log
        sc.pp.normalize_total(adata, target_sum=10000)
        sc.pp.log1p(adata)
        del adata.uns["log1p"]
    
        # Filter matrix to genes with protein sequences
        adata = adata[:, adata.var_names.isin(proteins)]
        # Calculate highly variable genes
        # 版本适用写法
        # scanpy.pp.highly_variable_genes(adata, 2000)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        
        adata.var.drop(["means", "dispersions", "dispersions_norm"], axis="columns", inplace=True)
        adata.uns["hvg"] = {}
        # del adata.uns["hvg"]
        return adata
    TACTiCS.norm_adata = customize_TACTiCS_norm_adata
    del customize_TACTiCS_norm_adata
    
    def customize_TACTiCS___init__(self, dict_A=None, dict_B=None, match_path=None, folder=None):
        if folder is not None:
            self.load(folder)
            return
    
        assert dict_A is not None and dict_B is not None and match_path is not None
    
        self.dict_A, self.dict_B = dict_A, dict_B
    
        with open(match_path, "rb") as f:
            proteins_A, proteins_B, self.gene_matches = pickle.load(f)
        # counts已经再外部读入
        self.adata_A = dict_A["counts"]
        self.adata_B = dict_B["counts"]
        dict_A["counts"] = self.adata_A.uns['path']
        dict_B["counts"] = self.adata_B.uns['path']
    
        def filter_cells(adata, adata_dict):
            if "filter_column" in adata_dict and "filter_values" in adata_dict:
                if adata_dict["filter_column"] is None or adata_dict["filter_values"] is None:
                    return adata
                return adata[adata.obs[adata_dict["filter_column"]].isin(adata_dict["filter_values"])]
            return adata
    
        self.adata_A = filter_cells(self.adata_A, self.dict_A)
        self.adata_B = filter_cells(self.adata_B, self.dict_B)
    
        def filter_entries(entry_to_genes, adata):
            gene_to_entry = entry_to_genes.explode("Gene Names")
            gene_to_entry["Entry"] = gene_to_entry.index
            gene_to_entry = gene_to_entry.set_index("Gene Names")
            gene_to_entry = gene_to_entry.sort_values(["Reviewed", "Entry"])
            gene_to_entry = gene_to_entry[~gene_to_entry.index.duplicated(keep="first")]
            gene_to_entry = gene_to_entry[gene_to_entry.index.isin(adata.var_names)]
    
            gene_to_entry = gene_to_entry.sort_index()
            genes = gene_to_entry.index.values
    
            entry_to_gene_idx = {}
            for i, entry in enumerate(gene_to_entry["Entry"]):
                if entry not in entry_to_gene_idx:
                    entry_to_gene_idx[entry] = []
                entry_to_gene_idx[entry].append(i)
    
            return genes, entry_to_gene_idx
    
        if "genes" in self.dict_A:
            names_A = pd.read_pickle(self.dict_A["genes"])
            adata_genes_A, entry_to_gene_A = filter_entries(names_A, self.adata_A)
        else:
            adata_genes_A = sorted(list(set(self.adata_A.var_names).intersection(set(proteins_A))))
            entry_to_gene_A = {}
            for protein in proteins_A:
                if protein in adata_genes_A:
                    entry_to_gene_A[protein] = [adata_genes_A.index(protein)]
        
        if "genes" in self.dict_B:
            names_B = pd.read_pickle(self.dict_B["genes"])
            adata_genes_B, entry_to_gene_B = filter_entries(names_B, self.adata_B)
        else:
            adata_genes_B = sorted(list(set(self.adata_B.var_names).intersection(set(proteins_B))))
            entry_to_gene_B = {}
            for protein in proteins_B:
                if protein in adata_genes_B:
                    entry_to_gene_B[protein] = [adata_genes_B.index(protein)]
    
        if type(self.gene_matches) is not scipy.sparse.coo_matrix:
            self.gene_matches = scipy.sparse.coo_matrix(self.gene_matches)
    
        rows = []
        cols = []
        data = []
        for i in range(len(self.gene_matches.data)):
            protein_A = proteins_A[self.gene_matches.row[i]]
            protein_B = proteins_B[self.gene_matches.col[i]]
            if protein_A in entry_to_gene_A and protein_B in entry_to_gene_B:
                for row in entry_to_gene_A[protein_A]:
                    for col in entry_to_gene_B[protein_B]:
                        rows.append(row)
                        cols.append(col)
                        data.append(self.gene_matches.data[i])
    
        self.gene_matches = scipy.sparse.coo_matrix((data, (rows, cols)), shape=(len(adata_genes_A), len(adata_genes_B)))
    
        # # Remove matches from genes that are not in adata_A or adata_B
        # if type(self.gene_matches) is not scipy.sparse.coo_matrix:
        #     self.gene_matches = scipy.sparse.coo_matrix(self.gene_matches)
        # for i in range(len(self.gene_matches.data)):
        #     if proteins_A[self.gene_matches.row[i]] not in self.adata_A.var_names or \
        #             proteins_B[self.gene_matches.col[i]] not in self.adata_B.var_names:
        #         self.gene_matches.data[i] = 0
        # self.gene_matches.eliminate_zeros()
    
        # Filter gene matches top-5
        k = 5
        self.gene_matches = utils.filter_top_k(self.gene_matches, k)
    
        # Normalize counts and calculate highly variable genes
        self.adata_A = self.norm_adata(self.adata_A, adata_genes_A)
        self.adata_B = self.norm_adata(self.adata_B, adata_genes_B)
    
        # Filter gene matches to highly variable genes + matches
        self.gene_matches = scipy.sparse.coo_matrix(self.gene_matches)
        for i in range(len(self.gene_matches.data)):
            if not self.adata_A.var.highly_variable.loc[adata_genes_A[self.gene_matches.row[i]]] and \
                    not self.adata_B.var.highly_variable.loc[adata_genes_B[self.gene_matches.col[i]]]:
                self.gene_matches.data[i] = 0
        self.gene_matches.eliminate_zeros()
    
        genes_A = sorted([adata_genes_A[i] for i in set(self.gene_matches.row)])
        genes_B = sorted([adata_genes_B[i] for i in set(self.gene_matches.col)])
    
        genes_idx_A = [genes_A.index(x) if x in genes_A else -1 for x in adata_genes_A]
        genes_idx_B = [genes_B.index(x) if x in genes_B else -1 for x in adata_genes_B]
    
        gene_matches_new = torch.zeros((len(genes_A), len(genes_B)), dtype=torch.float32)
        for i in range(len(self.gene_matches.data)):
            gene_matches_new[genes_idx_A[self.gene_matches.row[i]], genes_idx_B[self.gene_matches.col[i]]] = self.gene_matches.data[i]
        self.gene_matches = gene_matches_new
    
        # Filter counts according to matches
        self.adata_A = self.adata_A[:, genes_A]
        self.adata_B = self.adata_B[:, genes_B]
    
        # Calculate Z-score per gene
        sc.pp.scale(self.adata_A)
        sc.pp.scale(self.adata_B)
        self.adata_A.var.drop(["mean", "std"], axis="columns", inplace=True)
        self.adata_B.var.drop(["mean", "std"], axis="columns", inplace=True)
    
        self.adata_A.raw = self.adata_A
        self.adata_B.raw = self.adata_B
    
        self.n_classes_A = len(self.adata_A.obs[self.dict_A["column"]].cat.categories)
        self.n_classes_B = len(self.adata_B.obs[self.dict_B["column"]].cat.categories)
    
        self.model = TACTiCSNet(self.n_classes_A, self.n_classes_B, len(genes_A), len(genes_B), self.gene_matches)
        self.model.apply(utils.init_weights)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001, weight_decay=0.01)
    TACTiCS.__init__ = customize_TACTiCS___init__
    del customize_TACTiCS___init__
    
    def customize_TACTiCS_get_directional_conf(self, include_counts=False):
        # 版本不适用 sklearn.metrics.confusion_matrix
        from sklearn.metrics import confusion_matrix
        
        
        labels_A = set(self.adata_A.obs[self.dict_A["column"]].cat.categories)
        labels_B = set(self.adata_B.obs[self.dict_B["column"]].cat.categories)
        labels = np.array(sorted(list(labels_A.union(labels_B))))
        idx_A = [i for i in range(len(labels)) if labels[i] in set(labels_A)]
        idx_B = [i for i in range(len(labels)) if labels[i] in set(labels_B)]
        labels_A = labels[idx_A]
        labels_B = labels[idx_B]
    
        y_A = self.adata_A.obs[self.dict_A["column"]].values
        transfer_A = self.adata_A.obs[self.dict_B["name"]].values
        y_B = self.adata_B.obs[self.dict_B["column"]].values
        transfer_B = self.adata_B.obs[self.dict_A["name"]].values
    
        if include_counts:
            labels_A = [f"{x} ({self.adata_A.obs[self.dict_A['column']].value_counts().loc[x]})" for x in labels_A]
            labels_B = [f"{x} ({self.adata_B.obs[self.dict_B['column']].value_counts().loc[x]})" for x in labels_B]
    
        conf_matrix_A = confusion_matrix(y_A, transfer_A, normalize="true", labels=labels)
        conf_matrix_A = conf_matrix_A[idx_A, :]
        conf_matrix_A = conf_matrix_A[:, idx_B]
    
        conf_matrix_B = confusion_matrix(y_B, transfer_B, normalize="true", labels=labels)
        conf_matrix_B = conf_matrix_B[idx_B, :]
        conf_matrix_B = conf_matrix_B[:, idx_A]
    
        return labels_A, labels_B, conf_matrix_A, conf_matrix_B
    TACTiCS.get_directional_conf = customize_TACTiCS_get_directional_conf
    del customize_TACTiCS_get_directional_conf
    
    def customize_utils_get_batch(adata, column, n):
        total = len(adata)
        if n == total:
            x = torch.Tensor(adata.X)
            y = adata.obs[column].cat.codes
        else:
            weights = 1 / adata.obs[column].value_counts()
            cts = adata.obs[column]
            # 版本适用写法
            # weights = np.array(pd.Series.replace(cts, weights).values)
            weights = np.array((weights[cts.astype(str)].values))
            # return cts,weights
            weights = weights / sum(weights)
            idx = sorted(np.random.choice(total, n, replace=True, p=weights))
    
            x = torch.Tensor(adata[idx, :].X)
            # 版本适用写法
            # y = adata.obs[column].cat.codes[idx]
            y = adata.obs[column].cat.codes.iloc[idx]
    
        # 版本适用写法
        y = torch.Tensor(y.values).to(torch.long)
        return x, y
    utils.get_batch = customize_utils_get_batch
    del customize_utils_get_batch


In [5]:
p_model_root = Path('~/link/other_model/learn_TACTiCS').expanduser()
p_model_root

PosixPath('/public/workspace/licanchengup/link/other_model/learn_TACTiCS')

In [6]:
def get_TACTiCS_gene_path(sp,p = p_model_root.joinpath('TACTiCS_genes_path.csv')):
    p = Path(p)
    df_gens = pd.read_csv(p).query("sp == '{}'".format(str(sp))).reset_index(drop=True)
    assert df_gens.shape[0] == 1,"[Error] get {} item with sp = {}".format(df_gens.shape[0],sp)
    return p.parent.joinpath(df_gens.at[0,'path'])

def get_TACTiCS_protein_dist(sp_ref,sp_que,p = p_model_root.joinpath('TACTiCS_protein_dist.csv')):
    p = Path(p)
    df_dist = pd.read_csv(p).query("sp_ref == '{}' & sp_que == '{}'".format(
        sp_ref,sp_que)).reset_index(drop=True)
    assert df_dist.shape[0] == 1,"[Error] get {} item with {},{}".format(df_dist.shape[0],sp_ref,sp_que)
    return p.parent.joinpath(df_dist.at[0,'path'])


# [from func.py]---------------------------------------------------------------------------

def get_path_varmap(sp1,sp2,model):
    if model == 'TACTiCS':
        return get_TACTiCS_protein_dist(sp1,sp2)
    else:
        raise Exception("[Error] can not find varmap path for model = {}".format(model))

def get_type_counts_info(adatas, key_class, dsnames):
    type_counts_list = []
    for i in range(len(adatas)):
        type_counts_list.append(pd.value_counts(adatas[i].obs[key_class]))
    counts_info = pd.concat(type_counts_list, axis=1, keys=dsnames)
    return counts_info

def aligned_type(adatas, key_calss):
    adata1 = adatas[0].copy()
    adata2 = adatas[1].copy()
    counts_info = get_type_counts_info(
        adatas, key_calss, dsnames=["reference", "query"]
    )
    print("----raw----")
    print(counts_info)
    counts_info = counts_info.dropna(how="any")
    print("----new----")
    print(counts_info)

    com_type = counts_info.index.tolist()
    adata1 = adata1[adata1.obs[key_calss].isin(com_type)]
    adata2 = adata2[adata2.obs[key_calss].isin(com_type)]
    return adata1, adata2

# [run TACTiCS]---------------------------------------------------------------------------


def precess_after_TACTiCS(
    resdir, tissue_name, sp1, sp2, is_display=False, **kvargs
):
    resdir  = Path(resdir)
    model = kvargs.setdefault('model',None)
    if model is None:
        model = TACTiCS(folder=str(resdir.joinpath('model')))
    df_obs = pd.concat([model.adata_A.obs.loc[:,[model.dict_A['column']]].rename(columns={model.dict_A['column']:'cell_type'}),
    model.adata_B.obs.loc[:,[model.dict_B['column']]].rename(columns={model.dict_B['column']:'cell_type'})])
    df_obs['dataset'] = np.concatenate([
        np.full(model.adata_A.shape[0],'{}_{}'.format(tissue_name,sp1)),
        np.full(model.adata_B.shape[0],'{}_{}'.format(tissue_name,sp2))])
    
    df_obs['true_label'] = df_obs['cell_type']
    df_obs = df_obs.join(model.adata_B.obs.loc[:,[model.dict_A['name']]].rename(columns={model.dict_A['name']:'pre_label'}))
    df_obs['max_prob'] = np.nan
    df_obs['is_right'] = df_obs['true_label'] == df_obs['pre_label']
    
    adata = ad.AnnData(np.concatenate([model.adata_A.obsm["emb"], model.adata_B.obsm["emb"]], axis=0),
                       obs=df_obs)
    sc.pp.neighbors(adata, n_neighbors=30)
    sc.tl.umap(adata)
    adata.obs = adata.obs.join(pd.DataFrame(adata.obsm['X_umap'],index=adata.obs.index,columns='UMAP1,UMAP2'.split(',')))\
        .loc[:,'UMAP1,UMAP2,dataset,cell_type,true_label,pre_label,max_prob,is_right'.split(',')]
    adata.obs.to_csv(resdir.joinpath('obs.csv'),index=True)
    # plot
    resdir.joinpath('figs').mkdir(exist_ok=True,parents=True)
    sc.pl.umap(adata,color='cell_type',size=2,return_fig=True)\
        .savefig(resdir.joinpath('figs/umap_umap.png'))
    sc.pl.umap(adata,color='dataset',size=2,return_fig=True)\
        .savefig(resdir.joinpath('figs/umap_dataset.png'))


def run_TACTiCS(
    path_adata1,
    path_adata2,
    key_class1,
    key_class2,
    sp1,
    sp2,
    tissue_name,
    path_varmap,
    limite_func=lambda adata1,adata2: (adata1,adata2),
    aligned=False,
    resdir_tag=".",
    resdir=Path('.'), **kvargs
):
    """
    version:0.0.5
    kvargs:
        n_epochs: int
            default,50 见TACTiCS tutorial
    """

    # Parameter settings
    n_epochs = sum(kvargs.setdefault("n_epochs", [50]))

    # setting directory for results
    if len(resdir_tag) > 0:

        resdir_tag = "{}_{}-corss-{};{}".format(
            tissue_name, sp1, sp2, resdir_tag)
    else:
        resdir_tag = "{}_{}-corss-{}".format(tissue_name, sp1, sp2)

    resdir = resdir.joinpath(resdir_tag)

    # 终止 判断
    p_finish = resdir.joinpath("finish")
    if p_finish.exists():
        # precess_after_came(resdir,tissue_name,sp1, sp2)
        print(
            "[has finish]{} {}".format(
                time.strftime('%y%m%d-%H%M', time.localtime()),
                resdir.name)
        )
        return
    print(
        "[start]{} {}".format(
            time.strftime('%y%m%d-%H%M', time.localtime()),
            resdir.name

        ))
    # return

    figdir = resdir.joinpath("figs")
    sc.settings.figdir = figdir
    resdir.mkdir(parents=True, exist_ok=True)

    finish_content = ["[strat] {}".format(time.time())]

    # # setting

    dsnames = (
        '{}_{}'.format(
            tissue_name, sp1), '{}_{}'.format(
            tissue_name, sp2))
    dsn1, dsn2 = dsnames

    # load data
    adata_raw1 = ut.sc.load_adata(path_adata1)
    adata_raw2 = ut.sc.load_adata(path_adata2)
    adata_raw1.uns['path'] = str((path_adata1))
    adata_raw2.uns['path'] = str((path_adata2))

    key_class = key_class1
    if key_class not in adata_raw2.obs.columns:
        adata_raw2.obs[key_class] = ''
    adata_raw1.obs[key_class1] = pd.Categorical(adata_raw1.obs[key_class1])
    adata_raw2.obs[key_class2] = pd.Categorical(adata_raw2.obs[key_class2])
    
    # limite 进一步对adata进行限制，默认不操作直接返回
    adata_raw1, adata_raw2 = limite_func(adata_raw1, adata_raw2)

    # group_counts_unalign.csv
    pd.concat(
        [
            adata_raw1.obs[key_class1].value_counts(),
            adata_raw2.obs[key_class2].value_counts(),
        ],
        axis=1,
        keys=dsnames,
    ).to_csv(resdir.joinpath("group_counts_unalign.csv"), index=True)
    # align
    if aligned:
        adata_raw1, adata_raw2 = aligned_type(
            [adata_raw1, adata_raw2], key_calss=key_class1
        )

    # 保存obs ,即真正测试的细胞的mata
    adata_raw1.obs.to_csv(resdir.joinpath("obs_ref.csv"), index=True)
    adata_raw2.obs.to_csv(resdir.joinpath("obs_que.csv"), index=True)

    # group_counts.csv
    temp = pd.concat(
        [
            adata_raw1.obs[key_class1].value_counts(),
            adata_raw2.obs[key_class2].value_counts(),
        ],
        axis=1,
        keys=dsnames,
    )
    temp.to_csv(resdir.joinpath("group_counts.csv"), index=True)
    if temp.shape[0] < 2:
        # 错误标记
        print("[Error][group_counts no any item]")
        finish_content.append(
            "[Error][group_counts no any item] %f" % time.time()
        )
        p_finish.with_name("error").write_text("\n".join(finish_content))
        return

    print("cell count --> {}".format(sum([adata_raw1.shape[0]+adata_raw2.shape[0]])))
    
    kvargs.update({'path_adata1': str(path_adata1),
                   'path_adata2': str(path_adata2),
                   'key_class1': key_class1,
                   'key_class2': key_class2,
                   'sp1': sp1,
                   'sp2': sp2,
                   'tissue_name': tissue_name,
                   'path_varmap': str(path_varmap),
                   'aligned': aligned,
                   'resdir_tag': resdir_tag,
                   'resdir': str(resdir),
                   'n_epochs':n_epochs}
                  )
    
    resdir.joinpath("kvargs.json").write_text(json.dumps(kvargs))

    finish_content.append("[finish before run] {}".format(time.time()))
    
    dict_A = dict(
        name = sp1,
        counts=adata_raw1,
        genes= str(get_TACTiCS_gene_path(map_sp[sp1])),
        column = key_class1
    )
    dict_B = dict(
            name = sp2,
            counts=adata_raw2,
            genes= str(get_TACTiCS_gene_path(map_sp[sp2])),
            column = key_class2
        )
    # run
    model = TACTiCS(dict_A, dict_B, path_varmap)
    model.train(n_epochs=n_epochs)
    model.transfer()
    finish_content.append("[finish run] {}".format(time.time()))
    model.save(str(resdir.joinpath('model')))
    
    # 后处理
    precess_after_TACTiCS(resdir, tissue_name, sp1, sp2,model=model)
    finish_content.append("[finish after run] {}".format(time.time()))

    # 完成标记
    finish_content.append("[end] {}".format(time.time()))
    p_finish.write_text("\n".join(finish_content))

map_func_run_cross_species_models = {
    'TACTiCS': run_TACTiCS,

}

del run_TACTiCS
def run_cross_species_models(
    path_adata1,
    path_adata2,
    key_class1,
    key_class2,
    sp1,
    sp2,
    tissue_name,
    resdir,
    resdir_tag="",
    aligned=False,
    limite_func=lambda adata1, adata2: (adata1, adata2),
    models=''.split(','),
    **kvargs
):
    """
        models: TACTiCS
        kvargs:
        n_epochs:
            default,[50] 见TACTiCS tutorial
            stages,即res_0,res_1，res_2 的 epochs
            累加制，res_0,res_1，res_2,实际epochs分别为100,300,600
            故最终epochs为stages之和
            stages = kvargs.setdefault("n_epochs",[100, 200, 300])
        is_1v1: bool
            default,True TACTiCS仅one2one
    """
    
    for model in models:
        path_varmap = get_path_varmap(map_sp[sp1], map_sp[sp2], model=model)
        print('[path_varmap] {}\t{}'.format(model, Path(path_varmap).name))
        map_func_run_cross_species_models[model](
            path_adata1,
            path_adata2,
            key_class1,
            key_class2,
            sp1,
            sp2,
            tissue_name,
            path_varmap,
            aligned=aligned,
            resdir_tag=";".join([model, resdir_tag]),
            resdir=resdir,
            limite_func=limite_func,
            **kvargs,
        )


In [7]:
p_root = Path('~/link/csMAHN_publish').expanduser()
p_res = p_root.joinpath("res")
p_cache = p_root.joinpath("cache")

map_sp = {k: v for k, v in zip(
    'h,m,z,ma,c,x'.split(','),
    'human,mouse,zebrafish,macaque,chicken,xenopus'.split(',')
)}
map_sp_reverse = {v: k for k, v in map_sp.items()}

map_sp.update({k: v for k, v in zip(
    'hs,mm'.split(','),
    'human,mouse'.split(',')
)})

In [8]:
q_item = 'HCL_MCA,Retina'.split(',')
# q_item = 'HCL_MCA'.split(',')

df_para = pd.concat([pd.read_csv(p_cache.joinpath(
    'parameter_healthy_{}.csv'.format(i))).assign(mask=i)
    for i in q_item])
df_para['path_ref'] = df_para['path_ref'].apply(
    lambda x: p_cache.joinpath(x))
df_para['path_que'] = df_para['path_que'].apply(
    lambda x: p_cache.joinpath(x))

display(df_para.head(2),df_para.shape)

,tissue,sp_ref,path_ref,name_ref,sp_simple_ref,sp_que,path_que,name_que,sp_simple_que,key_cell_type,mask
0,Adrenal-Gland,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_adr,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_adr,m,CL,HCL_MCA
1,Bone-Marrow,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_bon,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_bon,m,CL,HCL_MCA


(21, 11)

In [9]:
for i, row in df_para.iterrows():
    # n_epochs = [2] # for test
    n_epochs = [50]
    run_cross_species_models(
        path_adata1=row['path_ref'],
        path_adata2=row['path_que'],
        key_class1=row['key_cell_type'],
        key_class2=row['key_cell_type'],
        sp1=row['sp_simple_ref'],
        sp2=row['sp_simple_que'],
        tissue_name=row['tissue'],
        resdir_tag="{name_ref}-map-{name_que};is_1v1=True".format(**row),
        resdir=p_res,
        n_epochs=n_epochs,
        aligned=True,
        models = ['TACTiCS'])
print("\n[finish][run]\n".center(100,'-'))

[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Adrenal-Gland_h-corss-m;TACTiCS;h_adr-map-m_adr;is_1v1=True
[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Bone-Marrow_h-corss-m;TACTiCS;h_bon-map-m_bon;is_1v1=True
[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Brain_h-corss-m;TACTiCS;h_bra-map-m_bra;is_1v1=True
[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Heart_h-corss-m;TACTiCS;h_hea-map-m_hea;is_1v1=True
[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Intestine_h-corss-m;TACTiCS;h_int-map-m_int;is_1v1=True
[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Kidney_h-corss-m;TACTiCS;h_kid-map-m_kid;is_1v1=True
[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Liver_h-corss-m;TACTiCS;h_liv-map-m_liv;is_1v1=True
[path_varmap] TACTiCS	human_mouse_dist.pkl
[has finish]241208-0011 Lung_h-corss-m;TACTiCS;h_lun-map-m_lun;is_1v1=True
[path_varmap] TACTiCS	human_mo

In [11]:
q_item = 'LC,RA'.split(',')

df_para = pd.concat([pd.read_csv(p_cache.joinpath(
    'parameter_{}.csv'.format(i))).assign(mask=i)
    for i in q_item])
df_para['path_ref'] = df_para['path_ref'].apply(
    lambda x: p_cache.joinpath(x))
df_para['path_que'] = df_para['path_que'].apply(
    lambda x: p_cache.joinpath(x))
df_para = df_para.reset_index(drop=True)
df_para

,tissue,sp_ref,path_ref,name_ref,sp_simple_ref,sp_que,path_que,name_que,sp_simple_que,key_cell_type,mask
0,LC,human,/public/workspace/licanchengup/link/csMAHN_pub...,LChDCs,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,LCmDCs,m,sub_cell_type,LC
1,LC,human,/public/workspace/licanchengup/link/csMAHN_pub...,LChNeu,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,LCmNeu,m,sub_cell_type,LC
2,LC,human,/public/workspace/licanchengup/link/csMAHN_pub...,LChMac,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,LCmMac,m,sub_cell_type,LC
3,LC,human,/public/workspace/licanchengup/link/csMAHN_pub...,LChall,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,LCmall,m,cell_type,LC
4,LC,human,/public/workspace/licanchengup/link/csMAHN_pub...,LChMono,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,LCmMono,m,sub_cell_type,LC
5,RA,human,/public/workspace/licanchengup/link/csMAHN_pub...,RAhfib,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,RAmfib,m,sub_cell_type,RA


In [12]:
for i, row in df_para.iterrows():
    # n_epochs = [2] # for test
    n_epochs = [50]
    run_cross_species_models(
        path_adata1=row['path_ref'],
        path_adata2=row['path_que'],
        key_class1=row['key_cell_type'],
        key_class2=row['key_cell_type'],
        sp1=row['sp_simple_ref'],
        sp2=row['sp_simple_que'],
        tissue_name=row['tissue'],
        resdir_tag="{name_ref}-map-{name_que};is_1v1=True".format(**row),
        resdir=p_res,
        n_epochs=n_epochs,
        aligned=False,
        models = ['TACTiCS'])
print("\n[finish][run]\n".center(100,'-'))

[path_varmap] TACTiCS	human_mouse_dist.pkl
[start]241208-0011 LC_h-corss-m;TACTiCS;LChDCs-map-LCmDCs;is_1v1=True
cell count --> 2656


/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:696: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}
/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:696: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}
/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/scanpy/preprocessing/_scale.py:318: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
/public/workspace/licanchengup/apps/miniconda3/envs/TACTiCS/lib/python3.12/site-packages/scanpy/preprocessing/_scale.py:318: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
  0%|          | 0

KeyboardInterrupt: 

In [10]:
print("\n[finish]\n".center(100,'-'))

---------------------------------------------
[finish]
---------------------------------------------
